In [1]:
from pathlib import Path

DATA = Path("/home/seowon1211/data1/conditioned_expression")
PREPARED = DATA / "prepared_controls"
STRUCTURE = DATA / "morgan_matched/morgan_binary.csv"
PATHWAY = DATA / "kegg_rat_20261102_034010_265aed"

RUN_TRAINING = False  # 지금은 학습하지 않음

for name, path in {
    "발현 데이터": PREPARED,
    "약물 구조": STRUCTURE,
    "Pathway": PATHWAY,
}.items():
    print(f"{name}: {'있음' if path.exists() else '경로 확인 필요'}")

발현 데이터: 있음
약물 구조: 있음
Pathway: 있음


In [2]:
import json
import numpy as np
import pandas as pd

# 큰 발현 배열은 읽기 전용으로 열기
vitro = np.load(PREPARED / "vitro_expression.npy", mmap_mode="r")
vivo = np.load(PREPARED / "vivo_expression.npy", mmap_mode="r")

probes = pd.read_csv(PREPARED / "probes.csv")
vitro_conditions = pd.read_csv(PREPARED / "vitro_conditions.csv")
vivo_conditions = pd.read_csv(PREPARED / "vivo_conditions.csv")
pairs = pd.read_csv(PREPARED / "pairs.csv.gz")

structure = pd.read_csv(STRUCTURE)
projection = np.load(PATHWAY / "projection.npy")
pathway_table = pd.read_csv(PATHWAY / "pathway_audit.csv")
provenance = json.loads((PATHWAY / "provenance.json").read_text())

# 발현 행·열과 조건·probe 수가 일치하는지
assert vitro.shape == (len(vitro_conditions), len(probes))
assert vivo.shape == (len(vivo_conditions), len(probes))

# Pathway 행렬의 probe 순서가 발현 데이터와 같은지
assert provenance["species"] == "Rattus norvegicus"
assert provenance["probe_order"] == probes["probe"].astype(str).tolist()
assert projection.shape == (len(probes), len(pathway_table))

print("In vitro 발현:", vitro.shape)
print("In vivo 발현:", vivo.shape)
print("화합물 수:", pairs["compound"].nunique())
print("입력–정답 pair 수:", len(pairs))
print("구조 표:", structure.shape)
print("Pathway 행렬:", projection.shape)
print("Probe–pathway 연결 수:", np.count_nonzero(projection))
print("\n기본 크기·종·probe 순서 검사 통과")

In vitro 발현: (1675, 3453)
In vivo 발현: (2224, 3453)
화합물 수: 140
입력–정답 pair 수: 19936
구조 표: (137, 2049)
Pathway 행렬: (3453, 321)
Probe–pathway 연결 수: 19254

기본 크기·종·probe 순서 검사 통과


In [3]:
# 1. 구조가 누락된 화합물 확인
compound_names = set(pairs["compound"].astype(str))
structure_names = set(structure["compound"].astype(str).str.strip())

missing = sorted(compound_names - structure_names)
print("구조 누락 화합물:", missing)

# 2. 최종 9조건 실험의 기존 결과
RESULTS = DATA / "sparse_pathway_gan_ddpm_20261102_062000_f4c529"
comparison_path = RESULTS / "comparison.csv"

assert comparison_path.is_file(), f"결과 경로 확인: {comparison_path}"

comparison = pd.read_csv(comparison_path)

display(comparison[
    [
        "family",
        "pathway_mode",
        "structure",
        "absolute_rmse",
        "response_rmse",
        "ratio_to_no_change",
        "selected_epoch",
    ]
])

# 3. 구조 누락 화합물이 어느 분할에 있는지 확인
split = pd.read_csv(RESULTS / "compound_split.csv")

print("\n구조 누락 화합물의 분할:")
display(split[split["compound"].isin(missing)])

print("학습 실행 여부:", RUN_TRAINING)

구조 누락 화합물: ['LPS', 'TNFÉø', 'tunicamycin']


,family,pathway_mode,structure,absolute_rmse,response_rmse,ratio_to_no_change,selected_epoch
0,MLP,none,True,0.337249,0.313829,0.950662,10
1,MLP,sparse,True,0.333559,0.309485,0.937502,10
2,MLP,random_sparse,True,0.335970,0.313035,0.948257,10
3,cGAN,none,True,0.379863,0.332292,1.006589,25
4,cGAN,sparse,True,0.388245,0.332922,1.008498,25
5,cGAN,random_sparse,True,0.410567,0.333861,1.011343,25
6,epsilon-DDPM,none,True,0.354855,0.330387,1.000820,30
7,epsilon-DDPM,sparse,True,0.356288,0.328952,0.996474,25
8,epsilon-DDPM,random_sparse,True,0.357838,0.331531,1.004286,30



구조 누락 화합물의 분할:


,compound,stage
0,LPS,outer_test
1,TNFÉø,inner_train
137,tunicamycin,inner_train


학습 실행 여부: False


In [4]:
import torch
from torch import nn

# 1. 약물 이름을 기준으로 구조 표 정리
structure_table = structure.copy()
structure_table["compound"] = (
    structure_table["compound"].astype(str).str.strip()
)

assert not structure_table["compound"].duplicated().any()
structure_table = structure_table.set_index("compound")

# 이름을 제외한 2048개 열이 유한한 0/1 값인지 확인
bits = structure_table.to_numpy(dtype=np.float32)

assert bits.shape[1] == 2048
assert np.isfinite(bits).all()
assert np.isin(bits, [0, 1]).all()

# 2. 구조 유무 표시 추가
structure_table["structure_present"] = 1.0

# 3. 발현 데이터의 약물 순서에 맞추고, 누락은 0으로 채움
compound_order = sorted(pairs["compound"].unique())

structure_array = (
    structure_table
    .reindex(compound_order)
    .fillna(0)
    .to_numpy(dtype=np.float32)
)

compound_to_index = {
    name: index
    for index, name in enumerate(compound_order)
}

structure_tensor = torch.tensor(structure_array)

# 4. 구조 encoder 정의 — 아직 학습하지 않음
structure_encoder = nn.Sequential(
    nn.Linear(2049, 64, bias=False),
    nn.SiLU(),
    nn.Dropout(0.1),
    nn.Linear(64, 32, bias=False),
    nn.SiLU(),
)

print("구조 입력:", tuple(structure_tensor.shape))
print("구조 encoder: 2049 → 64 → 32")
print("모델 정의 완료 — 학습·예측은 실행하지 않았습니다.")

구조 입력: (140, 2049)
구조 encoder: 2049 → 64 → 32
모델 정의 완료 — 학습·예측은 실행하지 않았습니다.


In [5]:
class PathwayEncoder(nn.Module):
    def __init__(self, projection, units=4, output_dim=32):
        super().__init__()

        # 양수인 위치만 연결로 사용
        edges = np.argwhere(projection > 0)

        self.n_pathways = projection.shape[1]
        self.units = units

        # 연결 위치는 고정: 학습하지 않는 값
        self.register_buffer(
            "probe_index",
            torch.tensor(edges[:, 0], dtype=torch.long),
        )
        self.register_buffer(
            "pathway_index",
            torch.tensor(edges[:, 1], dtype=torch.long),
        )

        # pathway별 연결 수
        fan_in = np.bincount(
            edges[:, 1],
            minlength=self.n_pathways,
        )
        assert (fan_in > 0).all(), "연결이 없는 pathway가 있습니다."

        # 연결마다 가중치 4개: 학습되는 값
        weights = torch.randn(len(edges), units)
        scale = torch.tensor(
            np.sqrt(fan_in[edges[:, 1]]),
            dtype=torch.float32,
        )
        self.edge_weights = nn.Parameter(
            weights / scale[:, None]
        )

        self.bias = nn.Parameter(
            torch.zeros(self.n_pathways, units)
        )

        self.readout = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(
                self.n_pathways * units,
                output_dim,
                bias=False,
            ),
            nn.SiLU(),
        )

    def forward(self, genes):
        # genes: [batch, 3453]의 표준화한 in vitro 발현
        values = genes[:, self.probe_index]

        weighted = (
            values[:, :, None]
            * self.edge_weights[None, :, :]
        )

        aggregated = genes.new_zeros(
            len(genes), self.n_pathways, self.units
        )
        aggregated = aggregated.index_add(
            1, self.pathway_index, weighted
        )

        nodes = torch.nn.functional.silu(
            aggregated + self.bias
        )

        return self.readout(nodes.flatten(start_dim=1))


pathway_encoder = PathwayEncoder(projection)

print("고정 연결 수:", len(pathway_encoder.probe_index))
print("Pathway 노드:", pathway_encoder.n_pathways, "×", pathway_encoder.units)
print("최종 pathway 특징: 32차원")
print("학습 parameter 수:",
      sum(p.numel() for p in pathway_encoder.parameters()))
print("정의 완료 — 학습은 실행하지 않았습니다.")

고정 연결 수: 19254
Pathway 노드: 321 × 4
최종 pathway 특징: 32차원
학습 parameter 수: 119388
정의 완료 — 학습은 실행하지 않았습니다.


In [7]:
from sklearn.preprocessing import StandardScaler

# 1. 기존 분할 사용 — 새로 섞지 않음
train_compounds = set(
    split.loc[split["stage"] == "inner_train", "compound"]
)
val_compounds = set(
    split.loc[split["stage"] == "inner_validation", "compound"]
)
test_compounds = set(
    split.loc[split["stage"] == "outer_test", "compound"]
)

assert not (train_compounds & val_compounds)
assert not (train_compounds & test_compounds)
assert not (val_compounds & test_compounds)

assert (
    train_compounds | val_compounds | test_compounds
) == set(pairs["compound"])

# 2. 내부 학습 화합물에 해당하는 pair만 선택
train_rows = np.flatnonzero(
    pairs["compound"].isin(train_compounds).to_numpy()
)

source_ids = pairs["condition_id_source"].to_numpy(dtype=int)
target_ids = pairs["condition_id_target"].to_numpy(dtype=int)

# 같은 조건이 여러 pair에 있어도 한 번만 사용
train_source_ids = np.unique(source_ids[train_rows])
train_target_ids = np.unique(target_ids[train_rows])

# 3. In vitro: 학습 조건으로만 평균·표준편차 계산
source_scaler = StandardScaler()
source_scaler.fit(vitro[train_source_ids])

vitro_scaled = source_scaler.transform(vitro).astype(np.float32)

# 4. In vivo: 학습 조건의 probe별 평균만 빼기
target_mean = vivo[train_target_ids].mean(axis=0).astype(np.float32)
vivo_centered = (vivo - target_mean).astype(np.float32)

print(
    "화합물 분할:",
    len(train_compounds),
    len(val_compounds),
    len(test_compounds),
)
print("표준화된 vitro:", vitro_scaled.shape)
print("중심화된 vivo:", vivo_centered.shape)
print("Probe별 target 평균:", target_mean.shape)
print("내부 학습용 전처리 완료 — 모델 학습은 하지 않았습니다.")

화합물 분할: 89 23 28
표준화된 vitro: (1675, 3453)
중심화된 vivo: (2224, 3453)
Probe별 target 평균: (3453,)
내부 학습용 전처리 완료 — 모델 학습은 하지 않았습니다.


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# 1. 각 pair의 입력(source)·예측 대상(target) 조건
features = {}

for side in ["source", "target"]:
    # 숫자 조건
    features[f"{side}_dose"] = np.log1p(
        pairs[f"dose_{side}"].to_numpy(dtype=float)
    )
    features[f"{side}_time"] = (
        pairs[f"time_h_{side}"].to_numpy(dtype=float)
    )

    # 범주 조건
    for key in ["unit", "level"]:
        features[f"{side}_{key}"] = (
            pairs[f"{key}_{side}"].astype(str).to_numpy()
        )

condition_features = pd.DataFrame(features)

numeric_columns = [
    column for column in condition_features
    if column.endswith(("_dose", "_time"))
]
category_columns = [
    column for column in condition_features
    if column not in numeric_columns
]

# 2. 숫자는 표준화, 범주는 one-hot
condition_encoder = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_columns),
    (
        "category",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
        category_columns,
    ),
])

# 내부 학습 pair로만 적합
condition_encoder.fit(condition_features.iloc[train_rows])

condition_array = condition_encoder.transform(
    condition_features
).astype(np.float32)

condition_tensor = torch.tensor(condition_array)

print("조건 입력:", tuple(condition_tensor.shape))
print("숫자 조건:", numeric_columns)
print("범주 조건:", category_columns)
print("조건 전처리 완료 — 모델 학습은 하지 않았습니다.")

조건 입력: (19936, 16)
숫자 조건: ['source_dose', 'source_time', 'target_dose', 'target_time']
범주 조건: ['source_unit', 'source_level', 'target_unit', 'target_level']
조건 전처리 완료 — 모델 학습은 하지 않았습니다.


In [9]:
class IVIVEMLP(nn.Module):
    def __init__(self, projection, condition_dim, use_pathway=True):
        super().__init__()
        self.use_pathway = use_pathway

        self.expression_encoder = nn.Sequential(
            nn.Linear(3453, 128),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.SiLU(),
        )

        self.structure_encoder = nn.Sequential(
            nn.Linear(2049, 64, bias=False),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32, bias=False),
            nn.SiLU(),
        )

        self.pathway_encoder = PathwayEncoder(projection)

        self.decoder = nn.Sequential(
            nn.Linear(64 + 32 + 32 + condition_dim, 64),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 3453),
        )

    def forward(self, genes, conditions, structures):
        expression = self.expression_encoder(genes)
        chemical = self.structure_encoder(structures)

        if self.use_pathway:
            biological = self.pathway_encoder(genes)
        else:
            biological = genes.new_zeros(len(genes), 32)

        combined = torch.cat(
            [expression, chemical, biological, conditions],
            dim=1,
        )
        return self.decoder(combined)


# 가중치 초기화를 재현 가능하게 설정
torch.manual_seed(42)

model = IVIVEMLP(
    projection=projection,
    condition_dim=condition_tensor.shape[1],
    use_pathway=True,
)

print("결합 차원: 64 + 32 + 32 + 16 = 144")
print("최종 출력: probe 발현 3453개")
print("Parameter 수:", sum(p.numel() for p in model.parameters()))
print("모델 정의 완료 — 학습하지 않았습니다.")

결합 차원: 64 + 32 + 32 + 16 = 144
최종 출력: probe 발현 3453개
Parameter 수: 936665
모델 정의 완료 — 학습하지 않았습니다.


In [10]:
from sklearn.base import clone

# 최종 재학습 대상: 내부 학습 89 + 검증 23개
outer_train_rows = np.flatnonzero(
    pairs["compound"].isin(
        train_compounds | val_compounds
    ).to_numpy()
)

# 내부 encoder와 별개로 생성
outer_condition_encoder = clone(condition_encoder)
outer_condition_encoder.fit(
    condition_features.iloc[outer_train_rows]
)

inner_names = condition_encoder.get_feature_names_out()
outer_names = outer_condition_encoder.get_feature_names_out()

print("내부 학습 조건 차원:", len(inner_names))
print("최종 재학습 조건 차원:", len(outer_names))
print(
    "최종 재학습에서 추가되는 조건:",
    sorted(set(outer_names) - set(inner_names)),
)

내부 학습 조건 차원: 16
최종 재학습 조건 차원: 17
최종 재학습에서 추가되는 조건: ['category__source_unit_mg/mL']


In [11]:
# 처음 4개 pair
check_rows = np.arange(4)

# 각 pair에 해당하는 발현·조건·구조 가져오기
genes_batch = torch.tensor(
    vitro_scaled[source_ids[check_rows]]
)

conditions_batch = condition_tensor[check_rows]

compound_indices = [
    compound_to_index[name]
    for name in pairs.iloc[check_rows]["compound"]
]
structures_batch = structure_tensor[compound_indices]

# 예측 경로만 확인: gradient 계산·가중치 업데이트 없음
model.eval()

with torch.no_grad():
    output = model(
        genes_batch,
        conditions_batch,
        structures_batch,
    )

assert output.shape == (4, 3453)
assert torch.isfinite(output).all()

print("발현 입력:", tuple(genes_batch.shape))
print("조건 입력:", tuple(conditions_batch.shape))
print("구조 입력:", tuple(structures_batch.shape))
print("모델 출력:", tuple(output.shape))
print("입출력 검사 통과 — 학습은 하지 않았습니다.")

발현 입력: (4, 3453)
조건 입력: (4, 16)
구조 입력: (4, 2049)
모델 출력: (4, 3453)
입출력 검사 통과 — 학습은 하지 않았습니다.


In [12]:
# 학습 pair에서 같은 target이 몇 번 등장하는지 계산
_, inverse, counts = np.unique(
    target_ids[train_rows],
    return_inverse=True,
    return_counts=True,
)

train_weights = (1.0 / counts[inverse]).astype(np.float32)
train_weights /= train_weights.mean()

# Tensor로 준비
source_tensor = torch.tensor(vitro_scaled)
target_tensor = torch.tensor(vivo_centered)
train_weight_tensor = torch.tensor(train_weights)

pair_compound_indices = np.array([
    compound_to_index[name]
    for name in pairs["compound"]
])

print("학습 pair 수:", len(train_rows))
print("가중치 평균:", float(train_weights.mean()))

학습 pair 수: 12672
가중치 평균: 0.9999999403953552


In [13]:
def train_one_epoch(model, optimizer):
    model.train()
    total_loss = 0.0

    # 학습 pair 순서를 무작위로 섞음
    order = torch.randperm(len(train_rows))

    for local_indices in order.split(128):
        rows = train_rows[local_indices.numpy()]

        genes = source_tensor[source_ids[rows]]
        conditions = condition_tensor[rows]
        structures = structure_tensor[
            pair_compound_indices[rows]
        ]
        truth = target_tensor[target_ids[rows]]

        prediction = model(genes, conditions, structures)

        # Pair마다 3453개 probe의 제곱오차 평균
        pair_mse = (prediction - truth).square().mean(dim=1)

        # Target 반복 횟수를 보정한 loss
        weights = train_weight_tensor[local_indices]
        loss = (pair_mse * weights).mean()

        if not torch.isfinite(loss):
            raise ValueError("학습 손실에 NaN/inf가 있습니다.")

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )
        optimizer.step()

        total_loss += loss.item() * len(rows)

    return total_loss / len(train_rows)


print("학습 함수 정의 완료 — 아직 학습하지 않았습니다.")

학습 함수 정의 완료 — 아직 학습하지 않았습니다.


In [14]:
@torch.no_grad()
def predict_targets(model, rows, batch_size=128):
    model.eval()  # Dropout 끄기

    ids = np.unique(target_ids[rows])
    lookup = {
        int(target_id): index
        for index, target_id in enumerate(ids)
    }

    sums = np.zeros((len(ids), 3453), dtype=np.float64)
    counts = np.zeros(len(ids), dtype=np.int64)

    for start in range(0, len(rows), batch_size):
        batch_rows = rows[start:start + batch_size]

        prediction = model(
            source_tensor[source_ids[batch_rows]],
            condition_tensor[batch_rows],
            structure_tensor[pair_compound_indices[batch_rows]],
        )

        # 중심화한 예측을 원래 발현 척도로 복원
        prediction = prediction.cpu().numpy() + target_mean

        if not np.isfinite(prediction).all():
            raise ValueError("예측에 NaN/inf가 있습니다.")

        positions = np.array([
            lookup[int(cid)]
            for cid in target_ids[batch_rows]
        ])

        np.add.at(sums, positions, prediction)
        np.add.at(counts, positions, 1)

    assert (counts > 0).all()

    averaged_prediction = (
        sums / counts[:, None]
    ).astype(np.float32)

    return ids, averaged_prediction


# 내부 검증에 해당하는 pair 위치
val_rows = np.flatnonzero(
    pairs["compound"].isin(val_compounds).to_numpy()
)

print("검증 pair 수:", len(val_rows))
print("검증 함수 정의 완료 — 아직 실행하지 않았습니다.")

검증 pair 수: 3312
검증 함수 정의 완료 — 아직 실행하지 않았습니다.


In [15]:
def evaluate_predictions(ids, prediction):
    truth = vivo[ids]

    # 1. 처리군·대조군 전체의 절대 발현 오차
    absolute_rmse = np.sqrt(
        np.mean((prediction.astype(np.float64) - truth) ** 2)
    )

    # 같은 화합물·단위·시간의 대조군 찾기
    controls = vivo_conditions.loc[
        vivo_conditions["level"] == "Control"
    ]

    control_keys = ["compound", "unit", "time_h"]
    assert not controls.duplicated(control_keys).any(), (
        "동일 조건에 대조군이 여러 개 있습니다."
    )

    control_map = (
        controls.set_index(control_keys)["condition_id"].to_dict()
    )

    prediction_lookup = {
        int(cid): index for index, cid in enumerate(ids)
    }

    response_errors = []
    zero_response_errors = []

    # 2. 처리군에 대해서만 반응 평가
    for index, cid in enumerate(ids):
        condition = vivo_conditions.iloc[int(cid)]

        if condition["level"] == "Control":
            continue

        key = (
            condition["compound"],
            condition["unit"],
            condition["time_h"],
        )
        control_id = control_map.get(key)

        if control_id is None or control_id not in prediction_lookup:
            raise ValueError(f"대조군 또는 대조군 예측 누락: {key}")

        control_index = prediction_lookup[control_id]

        actual_response = vivo[cid] - vivo[control_id]
        predicted_response = (
            prediction[index] - prediction[control_index]
        )

        response_errors.append(
            np.mean(
                (predicted_response.astype(np.float64)
                 - actual_response) ** 2
            )
        )
        zero_response_errors.append(
            np.mean(actual_response.astype(np.float64) ** 2)
        )

    if not response_errors:
        raise ValueError("반응을 평가할 처리군이 없습니다.")

    response_rmse = np.sqrt(np.mean(response_errors))
    zero_response_rmse = np.sqrt(np.mean(zero_response_errors))

    return {
        "absolute_rmse": float(absolute_rmse),
        "response_rmse": float(response_rmse),
        "zero_response_rmse": float(zero_response_rmse),
        "ratio_to_no_change": (
            float(response_rmse / zero_response_rmse)
            if zero_response_rmse > 0 else np.nan
        ),
    }


print("평가 함수 정의 완료 — 학습·평가는 실행하지 않았습니다.")

평가 함수 정의 완료 — 학습·평가는 실행하지 않았습니다.


In [16]:
# 내부 검증 데이터만 사용 — 테스트 데이터는 사용하지 않음
check_ids, check_prediction = predict_targets(model, val_rows)

check_metrics = evaluate_predictions(
    check_ids,
    check_prediction,
)

print("검증 target 수:", len(check_ids))
print("예측 배열:", check_prediction.shape)
print("\n미학습 모델의 동작 확인 점수:")
print(check_metrics)
print("\n학습이나 가중치 업데이트는 수행하지 않았습니다.")

검증 target 수: 368
예측 배열: (368, 3453)

미학습 모델의 동작 확인 점수:
{'absolute_rmse': 0.3240183453019862, 'response_rmse': 0.25482139420858696, 'zero_response_rmse': 0.2522472552292461, 'ratio_to_no_change': 1.0102048245361537}

학습이나 가중치 업데이트는 수행하지 않았습니다.


In [17]:
# 앞서 서버에서 불러온 기존 최종 결과만 표시
response_table = comparison.pivot(
    index="family",
    columns="pathway_mode",
    values="response_rmse",
).reindex(
    index=["MLP", "cGAN", "epsilon-DDPM"],
    columns=["none", "sparse", "random_sparse"],
)

response_table.columns = [
    "Pathway 없음",
    "실제 KEGG sparse",
    "무작위 sparse",
]

print("기존 학습 완료 결과 — 테스트 반응 RMSE")
display(response_table)

print("위 표는 저장된 기존 결과입니다.")
print("방금 계산한 미학습 검증 점수는 포함하지 않았습니다.")
print("학습 실행 설정:", RUN_TRAINING)

기존 학습 완료 결과 — 테스트 반응 RMSE


,Pathway 없음,실제 KEGG sparse,무작위 sparse
family,,,
MLP,0.313829,0.309485,0.313035
cGAN,0.332292,0.332922,0.333861
epsilon-DDPM,0.330387,0.328952,0.331531


위 표는 저장된 기존 결과입니다.
방금 계산한 미학습 검증 점수는 포함하지 않았습니다.
학습 실행 설정: False


In [18]:
class ModelContext(nn.Module):
    def __init__(self, projection, condition_dim, use_pathway=True):
        super().__init__()

        template = IVIVEMLP(
            projection,
            condition_dim,
            use_pathway=use_pathway,
        )

        self.use_pathway = use_pathway
        self.expression_encoder = template.expression_encoder
        self.structure_encoder = template.structure_encoder
        self.pathway_encoder = template.pathway_encoder

        # MLP decoder의 마지막 발현 출력층을 제외
        self.fusion = template.decoder[:3]

    def forward(self, genes, conditions, structures):
        expression = self.expression_encoder(genes)
        chemical = self.structure_encoder(structures)

        if self.use_pathway:
            biological = self.pathway_encoder(genes)
        else:
            biological = genes.new_zeros(len(genes), 32)

        combined = torch.cat(
            [expression, chemical, biological, conditions],
            dim=1,
        )
        return self.fusion(combined)

In [19]:
class ConditionalGenerator(nn.Module):
    def __init__(self, projection, condition_dim, use_pathway=True):
        super().__init__()

        self.context = ModelContext(
            projection, condition_dim, use_pathway
        )
        self.head = nn.Sequential(
            nn.Linear(64 + 16, 64),
            nn.SiLU(),
            nn.Linear(64, 3453),
        )

    def forward(self, genes, conditions, structures, noise):
        context = self.context(genes, conditions, structures)
        return self.head(torch.cat([context, noise], dim=1))


class ConditionalDiscriminator(nn.Module):
    def __init__(self, projection, condition_dim, use_pathway=True):
        super().__init__()

        self.context = ModelContext(
            projection, condition_dim, use_pathway
        )
        self.head = nn.Sequential(
            nn.Linear(64 + 3453, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, genes, conditions, structures, expression):
        context = self.context(genes, conditions, structures)
        combined = torch.cat([context, expression], dim=1)
        return self.head(combined).squeeze(1)


print("cGAN 정의 완료 — 인스턴스 생성·학습은 하지 않았습니다.")

cGAN 정의 완료 — 인스턴스 생성·학습은 하지 않았습니다.


In [20]:
class EpsilonDDPM(nn.Module):
    def __init__(
        self,
        projection,
        condition_dim,
        use_pathway=True,
        steps=100,
    ):
        super().__init__()

        self.context = ModelContext(
            projection, condition_dim, use_pathway
        )

        # 노이즈 단계 번호 → 특징 16개
        self.time_embedding = nn.Embedding(steps, 16)

        self.correction = nn.Sequential(
            nn.Linear(64 + 3453 + 16, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 3453),
        )

        # 기존 실험과 동일한 초기화
        nn.init.zeros_(self.correction[-1].weight)
        nn.init.zeros_(self.correction[-1].bias)

    def forward(
        self, genes, conditions, structures, noisy, timestep
    ):
        context = self.context(genes, conditions, structures)
        time_features = self.time_embedding(timestep)

        combined = torch.cat(
            [context, noisy, time_features],
            dim=1,
        )

        # 기존 구현의 identity skip
        return noisy + self.correction(combined)


def ddpm_schedule(steps=100):
    beta = torch.linspace(1e-4, 0.02, steps) * (1000 / steps)
    beta = beta.clamp(max=0.999)

    alpha = 1 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)

    return beta, alpha, alpha_bar


print("DDPM 정의 완료 — 인스턴스 생성·학습은 하지 않았습니다.")

DDPM 정의 완료 — 인스턴스 생성·학습은 하지 않았습니다.


In [21]:
import torch.nn.functional as F

def train_cgan_one_epoch(
    generator,
    discriminator,
    generator_optimizer,
    discriminator_optimizer,
    batch_size=128,
):
    generator.train()
    discriminator.train()

    total_g = 0.0
    total_d = 0.0

    order = torch.randperm(len(train_rows))

    for local_indices in order.split(batch_size):
        rows = train_rows[local_indices.numpy()]

        genes = source_tensor[source_ids[rows]]
        conditions = condition_tensor[rows]
        structures = structure_tensor[pair_compound_indices[rows]]
        truth = target_tensor[target_ids[rows]]
        weights = train_weight_tensor[local_indices]

        # ── 1. 판별기 학습 ──
        for parameter in discriminator.parameters():
            parameter.requires_grad_(True)

        noise = torch.randn(len(rows), 16)

        # detach: 이 단계에서는 생성기 가중치를 바꾸지 않음
        fake = generator(
            genes, conditions, structures, noise
        ).detach()

        real_logits = discriminator(
            genes, conditions, structures, truth
        )
        fake_logits = discriminator(
            genes, conditions, structures, fake
        )

        real_loss = F.binary_cross_entropy_with_logits(
            real_logits,
            torch.ones_like(real_logits),
            reduction="none",
        )
        fake_loss = F.binary_cross_entropy_with_logits(
            fake_logits,
            torch.zeros_like(fake_logits),
            reduction="none",
        )

        loss_d = (0.5 * (real_loss + fake_loss) * weights).mean()

        if not torch.isfinite(loss_d):
            raise ValueError("판별기 손실에 NaN/inf가 있습니다.")

        discriminator_optimizer.zero_grad()
        loss_d.backward()
        nn.utils.clip_grad_norm_(discriminator.parameters(), 1.0)
        discriminator_optimizer.step()

        # ── 2. 생성기 학습 ──
        for parameter in discriminator.parameters():
            parameter.requires_grad_(False)

        noise = torch.randn(len(rows), 16)
        fake = generator(genes, conditions, structures, noise)

        fake_logits = discriminator(
            genes, conditions, structures, fake
        )

        # 생성기는 판별기가 '실제=1'이라고 판단하도록 학습
        loss_g = (
            F.binary_cross_entropy_with_logits(
                fake_logits,
                torch.ones_like(fake_logits),
                reduction="none",
            ) * weights
        ).mean()

        if not torch.isfinite(loss_g):
            raise ValueError("생성기 손실에 NaN/inf가 있습니다.")

        generator_optimizer.zero_grad()
        loss_g.backward()
        nn.utils.clip_grad_norm_(generator.parameters(), 1.0)
        generator_optimizer.step()

        total_g += loss_g.item() * len(rows)
        total_d += loss_d.item() * len(rows)

    return {
        "generator_loss": total_g / len(train_rows),
        "discriminator_loss": total_d / len(train_rows),
    }


print("cGAN 학습 함수 정의 완료 — 학습은 실행하지 않았습니다.")

cGAN 학습 함수 정의 완료 — 학습은 실행하지 않았습니다.


In [22]:
def train_ddpm_one_epoch(
    model,
    optimizer,
    steps=100,
    batch_size=128,
):
    model.train()
    total_loss = 0.0

    _, _, alpha_bar = ddpm_schedule(steps)

    order = torch.randperm(len(train_rows))

    for local_indices in order.split(batch_size):
        rows = train_rows[local_indices.numpy()]

        genes = source_tensor[source_ids[rows]]
        conditions = condition_tensor[rows]
        structures = structure_tensor[pair_compound_indices[rows]]

        # 학습 평균을 뺀 실제 in vivo 발현
        truth = target_tensor[target_ids[rows]]
        weights = train_weight_tensor[local_indices]

        # Pair마다 노이즈 단계와 노이즈를 무작위로 선택
        timestep = torch.randint(steps, (len(rows),))
        noise = torch.randn_like(truth)

        # 실제 발현과 노이즈를 섞음
        noisy = (
            alpha_bar[timestep, None].sqrt() * truth
            + (1 - alpha_bar[timestep, None]).sqrt() * noise
        )

        # 모델은 발현이 아니라 '섞었던 노이즈'를 예측
        predicted_noise = model(
            genes,
            conditions,
            structures,
            noisy,
            timestep,
        )

        pair_mse = (predicted_noise - noise).square().mean(dim=1)
        loss = (pair_mse * weights).mean()

        if not torch.isfinite(loss):
            raise ValueError("DDPM 손실에 NaN/inf가 있습니다.")

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(rows)

    return total_loss / len(train_rows)


print("DDPM 학습 함수 정의 완료 — 학습은 실행하지 않았습니다.")

DDPM 학습 함수 정의 완료 — 학습은 실행하지 않았습니다.


In [23]:
@torch.no_grad()
def generate_pair_mean(
    family,
    model,
    genes,
    conditions,
    structures,
    samples=4,
    steps=100,
    seed=20042,
):
    if family not in {"cgan", "ddpm"}:
        raise ValueError("family는 'cgan' 또는 'ddpm'입니다.")

    model.eval()
    rng = torch.Generator().manual_seed(seed)

    # 같은 입력 조건에서 여러 샘플 생성
    context = model.context(genes, conditions, structures)
    batch_size = len(context)
    context = context.repeat_interleave(samples, dim=0)

    if family == "cgan":
        noise = torch.randn(
            len(context), 16, generator=rng
        )

        generated = model.head(
            torch.cat([context, noise], dim=1)
        )

    else:
        beta, alpha, alpha_bar = ddpm_schedule(steps)

        # 무작위 노이즈에서 시작
        generated = torch.randn(
            len(context), 3453, generator=rng
        )

        # 99 → 0의 전체 역과정
        for step in reversed(range(steps)):
            timestep = torch.full(
                (len(context),),
                step,
                dtype=torch.long,
            )

            time_features = model.time_embedding(timestep)

            predicted_noise = generated + model.correction(
                torch.cat(
                    [context, generated, time_features],
                    dim=1,
                )
            )

            generated = (
                generated
                - beta[step]
                / (1 - alpha_bar[step]).sqrt()
                * predicted_noise
            ) / alpha[step].sqrt()

            if step > 0:
                variance = (
                    beta[step]
                    * (1 - alpha_bar[step - 1])
                    / (1 - alpha_bar[step])
                )

                generated = generated + variance.sqrt() * torch.randn(
                    generated.shape,
                    generator=rng,
                )

    if not torch.isfinite(generated).all():
        raise ValueError("生成結果にNaN/infがあります。")

    # [pair数 × samples, 3453] → [pair数, samples, 3453]
    generated = generated.reshape(batch_size, samples, 3453)

    # 中心化した発現の生成平均を返す
    return generated.mean(dim=1)


print("生成関数の定義完了 — 生成・学習は実行していません。")

生成関数の定義完了 — 生成・学習は実行していません。


In [24]:
@torch.no_grad()
def predict_generative_targets(
    family,
    model,
    rows,
    batch_size=128,
    samples=4,
    steps=100,
    seed=20042,
):
    ids = np.unique(target_ids[rows])

    lookup = {
        int(cid): index
        for index, cid in enumerate(ids)
    }

    sums = np.zeros((len(ids), 3453), dtype=np.float64)
    counts = np.zeros(len(ids), dtype=np.int64)

    for start in range(0, len(rows), batch_size):
        batch_rows = rows[start:start + batch_size]

        # 각 pair에서 생성 샘플 4개의 평균
        centered_prediction = generate_pair_mean(
            family=family,
            model=model,
            genes=source_tensor[source_ids[batch_rows]],
            conditions=condition_tensor[batch_rows],
            structures=structure_tensor[
                pair_compound_indices[batch_rows]
            ],
            samples=samples,
            steps=steps,
            seed=seed + start,
        )

        # 학습 target 평균을 더해 원래 척도로 복원
        prediction = (
            centered_prediction.cpu().numpy() + target_mean
        )

        positions = np.array([
            lookup[int(cid)]
            for cid in target_ids[batch_rows]
        ])

        # 동일 target에 연결된 여러 source의 예측을 모음
        np.add.at(sums, positions, prediction)
        np.add.at(counts, positions, 1)

    assert (counts > 0).all()

    prediction = (
        sums / counts[:, None]
    ).astype(np.float32)

    return ids, prediction


print("생성모델 예측 함수 정의 완료 — 실행하지 않았습니다.")

생성모델 예측 함수 정의 완료 — 실행하지 않았습니다.


In [25]:
RANDOM_RESULTS = DATA / (
    "pathway_random_control_20261102_060433_3ca44d"
)

mask_path = RANDOM_RESULTS / "random_membership.npy"
audit_path = RANDOM_RESULTS / "randomization_audit.json"

assert mask_path.is_file(), f"파일 확인: {mask_path}"
assert audit_path.is_file(), f"파일 확인: {audit_path}"

random_projection = np.load(mask_path)
random_audit = json.loads(audit_path.read_text())

# 저장된 값이 0/1인지 검사
assert np.isin(random_projection, [0, 1]).all()
assert random_projection.shape == projection.shape

real_mask = projection > 0
random_mask = random_projection > 0

# Probe별·pathway별 연결 수가 유지됐는지 확인
np.testing.assert_array_equal(
    real_mask.sum(axis=0),
    random_mask.sum(axis=0),
)
np.testing.assert_array_equal(
    real_mask.sum(axis=1),
    random_mask.sum(axis=1),
)

# 원본 연결과 겹치는 비율
overlap = (
    np.count_nonzero(real_mask & random_mask)
    / np.count_nonzero(real_mask)
)

print("무작위 mask:", random_projection.shape)
print("연결 수:", np.count_nonzero(random_mask))
print("원래 KEGG 연결과 겹치는 비율:", f"{overlap:.2%}")
print("연결 수 보존 검사 통과 — 학습은 하지 않았습니다.")

무작위 mask: (3453, 321)
연결 수: 19254
원래 KEGG 연결과 겹치는 비율: 16.64%
연결 수 보존 검사 통과 — 학습은 하지 않았습니다.


In [26]:
import random

def build_model(family, pathway_mode, seed=42, steps=100):
    # 모델 초기화 재현을 위한 seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if pathway_mode == "none":
        membership = projection
        use_pathway = False
    elif pathway_mode == "sparse":
        membership = projection
        use_pathway = True
    elif pathway_mode == "random_sparse":
        membership = random_projection
        use_pathway = True
    else:
        raise ValueError(f"알 수 없는 pathway 조건: {pathway_mode}")

    condition_dim = condition_tensor.shape[1]

    if family == "mlp":
        model = IVIVEMLP(
            membership, condition_dim, use_pathway
        )
        return model, None

    if family == "cgan":
        generator = ConditionalGenerator(
            membership, condition_dim, use_pathway
        )

        # 기존 코드처럼 생성기 optimizer를 먼저 생성
        generator_optimizer = torch.optim.AdamW(
            generator.parameters(), lr=0.001, weight_decay=0.01
        )

        discriminator = ConditionalDiscriminator(
            membership, condition_dim, use_pathway
        )

        return generator, discriminator

    if family == "ddpm":
        model = EpsilonDDPM(
            membership, condition_dim, use_pathway, steps
        )
        return model, None

    raise ValueError(f"알 수 없는 모델: {family}")


print("모델 선택 함수 정의 완료 — 호출·학습하지 않았습니다.")

모델 선택 함수 정의 완료 — 호출·학습하지 않았습니다.


In [27]:
def build_model(family, pathway_mode, seed=42, steps=100):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if pathway_mode not in {"none", "sparse", "random_sparse"}:
        raise ValueError(f"알 수 없는 조건: {pathway_mode}")

    membership = (
        random_projection
        if pathway_mode == "random_sparse"
        else projection
    )
    use_pathway = pathway_mode != "none"
    condition_dim = condition_tensor.shape[1]

    if family == "mlp":
        model = IVIVEMLP(
            membership, condition_dim, use_pathway
        )
        discriminator = None

    elif family == "cgan":
        model = ConditionalGenerator(
            membership, condition_dim, use_pathway
        )
        discriminator = ConditionalDiscriminator(
            membership, condition_dim, use_pathway
        )

    elif family == "ddpm":
        model = EpsilonDDPM(
            membership, condition_dim, use_pathway, steps
        )
        discriminator = None

    else:
        raise ValueError(f"알 수 없는 모델: {family}")

    return model, discriminator


def build_optimizers(model, discriminator=None):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.001,
        weight_decay=0.01,
    )

    discriminator_optimizer = None

    if discriminator is not None:
        discriminator_optimizer = torch.optim.AdamW(
            discriminator.parameters(),
            lr=0.001,
            weight_decay=0.01,
        )

    return optimizer, discriminator_optimizer


print("모델·optimizer 준비 함수 정의 완료")
print("함수 호출·학습은 하지 않았습니다.")

모델·optimizer 준비 함수 정의 완료
함수 호출·학습은 하지 않았습니다.


In [28]:
import copy

def fit_inner(family, pathway_mode, epochs=30, check_every=5):
    model, discriminator = build_model(
        family, pathway_mode, seed=42
    )

    optimizer, discriminator_optimizer = build_optimizers(
        model, discriminator
    )

    history = []
    best_rmse = float("inf")
    best_epoch = None
    best_state = None

    for epoch in range(1, epochs + 1):
        # 1. 한 epoch 학습
        if family == "mlp":
            loss = train_one_epoch(model, optimizer)

        elif family == "cgan":
            losses = train_cgan_one_epoch(
                model,
                discriminator,
                optimizer,
                discriminator_optimizer,
            )
            loss = losses["generator_loss"]

        else:
            loss = train_ddpm_one_epoch(model, optimizer)

        # 2. 정해진 epoch에 내부 검증
        should_validate = (
            epoch == 1
            or epoch % check_every == 0
            or epoch == epochs
        )

        if not should_validate:
            continue

        if family == "mlp":
            ids, prediction = predict_targets(model, val_rows)
        else:
            ids, prediction = predict_generative_targets(
                family,
                model,
                val_rows,
                samples=4,
                steps=100,
                seed=20042,
            )

        metrics = evaluate_predictions(ids, prediction)

        # 3. 절대 발현 오차가 가장 낮은 epoch 선택
        if metrics["absolute_rmse"] < best_rmse:
            best_rmse = metrics["absolute_rmse"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

        history.append({
            "epoch": epoch,
            "training_loss": loss,
            **metrics,
            "best_epoch": best_epoch,
        })

        print(
            f"{family}/{pathway_mode} "
            f"epoch={epoch}/{epochs} "
            f"val_absolute={metrics['absolute_rmse']:.5f} "
            f"val_response={metrics['response_rmse']:.5f} "
            f"best_epoch={best_epoch}",
            flush=True,
        )

    return {
        "best_epoch": best_epoch,
        "best_state": best_state,
        "history": pd.DataFrame(history),
    }


print("내부 학습·검증 함수 정의 완료 — 실행하지 않았습니다.")

내부 학습·검증 함수 정의 완료 — 실행하지 않았습니다.


In [29]:
def prepare_refit_data():
    # 1. 최종 학습·테스트 pair 위치
    refit_compounds = train_compounds | val_compounds

    refit_rows = np.flatnonzero(
        pairs["compound"].isin(refit_compounds).to_numpy()
    )
    test_rows = np.flatnonzero(
        pairs["compound"].isin(test_compounds).to_numpy()
    )

    assert not set(refit_rows) & set(test_rows)

    # 2. 고유 학습 조건
    refit_source_ids = np.unique(source_ids[refit_rows])
    refit_target_ids = np.unique(target_ids[refit_rows])

    # 3. 발현 전처리를 112개 화합물로 다시 적합
    refit_source_scaler = StandardScaler().fit(
        vitro[refit_source_ids]
    )
    refit_source = refit_source_scaler.transform(
        vitro
    ).astype(np.float32)

    refit_target_mean = (
        vivo[refit_target_ids].mean(axis=0).astype(np.float32)
    )
    refit_target = (
        vivo - refit_target_mean
    ).astype(np.float32)

    # 4. 조건 encoder도 다시 적합
    refit_condition_encoder = clone(condition_encoder)
    refit_condition_encoder.fit(
        condition_features.iloc[refit_rows]
    )
    refit_conditions = refit_condition_encoder.transform(
        condition_features
    ).astype(np.float32)

    # 5. 최종 학습 subset에서 target별 pair 가중치 재계산
    _, inverse, counts = np.unique(
        target_ids[refit_rows],
        return_inverse=True,
        return_counts=True,
    )
    weights = (1.0 / counts[inverse]).astype(np.float32)
    weights /= weights.mean()

    return {
        "train_rows": refit_rows,
        "test_rows": test_rows,
        "source_tensor": torch.tensor(refit_source),
        "target_tensor": torch.tensor(refit_target),
        "condition_tensor": torch.tensor(refit_conditions),
        "train_weights": torch.tensor(weights),
        "target_mean": refit_target_mean,
        "source_scaler": refit_source_scaler,
        "condition_encoder": refit_condition_encoder,
    }


print("최종 재학습 데이터 준비 함수 정의 완료")
print("전처리 재계산·모델 학습은 아직 실행하지 않았습니다.")

최종 재학습 데이터 준비 함수 정의 완료
전처리 재계산·모델 학습은 아직 실행하지 않았습니다.


In [30]:
def build_model(
    family,
    pathway_mode,
    seed=42,
    steps=100,
    condition_dim=None,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if pathway_mode not in {"none", "sparse", "random_sparse"}:
        raise ValueError(f"알 수 없는 조건: {pathway_mode}")

    membership = (
        random_projection
        if pathway_mode == "random_sparse"
        else projection
    )
    use_pathway = pathway_mode != "none"

    # 기본은 내부 학습용 차원
    if condition_dim is None:
        condition_dim = condition_tensor.shape[1]

    if family == "mlp":
        model = IVIVEMLP(
            membership, condition_dim, use_pathway
        )
        discriminator = None

    elif family == "cgan":
        model = ConditionalGenerator(
            membership, condition_dim, use_pathway
        )
        discriminator = ConditionalDiscriminator(
            membership, condition_dim, use_pathway
        )

    elif family == "ddpm":
        model = EpsilonDDPM(
            membership, condition_dim, use_pathway, steps
        )
        discriminator = None

    else:
        raise ValueError(f"알 수 없는 모델: {family}")

    return model, discriminator


# 기존 내부 학습 데이터도 같은 형식으로 묶기
inner_data = {
    "train_rows": train_rows,
    "validation_rows": val_rows,
    "source_tensor": source_tensor,
    "target_tensor": target_tensor,
    "condition_tensor": condition_tensor,
    "train_weights": train_weight_tensor,
    "target_mean": target_mean,
    "source_scaler": source_scaler,
    "condition_encoder": condition_encoder,
}

print("내부 학습 조건 차원:",
      inner_data["condition_tensor"].shape[1])
print("모델 준비 함수 교체 완료 — 학습하지 않았습니다.")

내부 학습 조건 차원: 16
모델 준비 함수 교체 완료 — 학습하지 않았습니다.


In [31]:
def train_stage_epoch(
    family,
    model,
    optimizer,
    data,
    discriminator=None,
    discriminator_optimizer=None,
    batch_size=128,
    steps=100,
):
    model.train()
    if discriminator is not None:
        discriminator.train()

    rows_all = data["train_rows"]
    order = torch.randperm(len(rows_all))
    total_loss = 0.0

    _, _, alpha_bar = ddpm_schedule(steps)

    for local in order.split(batch_size):
        rows = rows_all[local.numpy()]

        genes = data["source_tensor"][source_ids[rows]]
        conditions = data["condition_tensor"][rows]
        structures = structure_tensor[pair_compound_indices[rows]]
        truth = data["target_tensor"][target_ids[rows]]
        weights = data["train_weights"][local]

        if family == "mlp":
            prediction = model(genes, conditions, structures)
            pair_loss = (prediction - truth).square().mean(dim=1)
            loss = (pair_loss * weights).mean()

        elif family == "ddpm":
            timestep = torch.randint(steps, (len(rows),))
            noise = torch.randn_like(truth)

            noisy = (
                alpha_bar[timestep, None].sqrt() * truth
                + (1 - alpha_bar[timestep, None]).sqrt() * noise
            )

            prediction = model(
                genes, conditions, structures, noisy, timestep
            )
            pair_loss = (prediction - noise).square().mean(dim=1)
            loss = (pair_loss * weights).mean()

        elif family == "cgan":
            if discriminator is None or discriminator_optimizer is None:
                raise ValueError("cGAN 판별기와 optimizer가 필요합니다.")

            # 판별기 업데이트
            for p in discriminator.parameters():
                p.requires_grad_(True)

            fake = model(
                genes, conditions, structures,
                torch.randn(len(rows), 16),
            ).detach()

            real_logits = discriminator(
                genes, conditions, structures, truth
            )
            fake_logits = discriminator(
                genes, conditions, structures, fake
            )

            loss_d = (
                0.5 * (
                    F.binary_cross_entropy_with_logits(
                        real_logits,
                        torch.ones_like(real_logits),
                        reduction="none",
                    )
                    + F.binary_cross_entropy_with_logits(
                        fake_logits,
                        torch.zeros_like(fake_logits),
                        reduction="none",
                    )
                ) * weights
            ).mean()

            if not torch.isfinite(loss_d):
                raise ValueError("판별기 손실에 NaN/inf가 있습니다.")

            discriminator_optimizer.zero_grad()
            loss_d.backward()
            nn.utils.clip_grad_norm_(discriminator.parameters(), 1.0)
            discriminator_optimizer.step()

            # 생성기 업데이트
            for p in discriminator.parameters():
                p.requires_grad_(False)

            fake = model(
                genes, conditions, structures,
                torch.randn(len(rows), 16),
            )
            fake_logits = discriminator(
                genes, conditions, structures, fake
            )

            loss = (
                F.binary_cross_entropy_with_logits(
                    fake_logits,
                    torch.ones_like(fake_logits),
                    reduction="none",
                ) * weights
            ).mean()

        else:
            raise ValueError(f"알 수 없는 모델: {family}")

        if not torch.isfinite(loss):
            raise ValueError("학습 손실에 NaN/inf가 있습니다.")

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(rows)

    return total_loss / len(rows_all)


print("단계별 공통 학습 함수 정의 완료 — 실행하지 않았습니다.")

단계별 공통 학습 함수 정의 완료 — 실행하지 않았습니다.


In [32]:
@torch.no_grad()
def predict_stage(
    family,
    model,
    data,
    rows,
    batch_size=128,
    samples=4,
    steps=100,
    seed=20042,
):
    model.eval()

    ids = np.unique(target_ids[rows])
    lookup = {
        int(cid): index
        for index, cid in enumerate(ids)
    }

    sums = np.zeros((len(ids), 3453), dtype=np.float64)
    counts = np.zeros(len(ids), dtype=np.int64)

    for start in range(0, len(rows), batch_size):
        batch_rows = rows[start:start + batch_size]

        genes = data["source_tensor"][source_ids[batch_rows]]
        conditions = data["condition_tensor"][batch_rows]
        structures = structure_tensor[
            pair_compound_indices[batch_rows]
        ]

        if family == "mlp":
            centered_prediction = model(
                genes, conditions, structures
            )
        else:
            centered_prediction = generate_pair_mean(
                family=family,
                model=model,
                genes=genes,
                conditions=conditions,
                structures=structures,
                samples=samples,
                steps=steps,
                seed=seed + start,
            )

        # 今回の学習段階で求めた平均を足す
        prediction = (
            centered_prediction.cpu().numpy()
            + data["target_mean"]
        )

        if not np.isfinite(prediction).all():
            raise ValueError("예측에 NaN/inf가 있습니다.")

        positions = np.array([
            lookup[int(cid)]
            for cid in target_ids[batch_rows]
        ])

        np.add.at(sums, positions, prediction)
        np.add.at(counts, positions, 1)

    assert (counts > 0).all()

    prediction = (
        sums / counts[:, None]
    ).astype(np.float32)

    return ids, prediction


print("단계별 공통 예측 함수 정의 완료 — 실행하지 않았습니다.")

단계별 공통 예측 함수 정의 완료 — 실행하지 않았습니다.
